In [16]:
# Celda 1 - Conexión e imports
from dao.mongo_dao import MongoDAO
from dao.municipio_dao import MunicipioDAO
from bson import ObjectId

mongo = MongoDAO()
db = mongo.connect()
municipio_dao = MunicipioDAO(mongo)
print("Conectado a DB:", mongo.db.name)

Conectado a DB: civictech


In [17]:
# Celda 2 - Listar municipios en tabla simple (sin pandas)
def print_table(rows, headers):
    # calcular anchos
    cols = list(zip(*rows)) if rows else [[] for _ in headers]
    widths = []
    for i, h in enumerate(headers):
        col = cols[i] if i < len(cols) else []
        max_cell = max([len(str(x)) for x in col], default=0)
        widths.append(max(len(h), max_cell))
    # cabecera
    header_line = " | ".join(h.ljust(widths[i]) for i, h in enumerate(headers))
    sep = "-+-".join("-" * widths[i] for i in range(len(headers)))
    print(header_line)
    print(sep)
    # filas
    for r in rows:
        print(" | ".join(str(r[i]).ljust(widths[i]) if i < len(r) else "".ljust(widths[i]) for i in range(len(headers))))

municipios = municipio_dao.listar(limit=200)
if municipios:
    rows = []
    for m in municipios:
        rows.append([
            str(m.get("_id")),
            m.get("nombre", ""),
            m.get("provincia", ""),
            m.get("pais", ""),
            m.get("codigo_municipio", ""),
            (m.get("contacto") or {}).get("email", "")
        ])
    headers = ["_id", "nombre", "provincia", "pais", "codigo_municipio", "email_contacto"]
    print_table(rows, headers)
else:
    print("No se encontraron municipios.")



_id                      | nombre           | provincia | pais      | codigo_municipio | email_contacto            
-------------------------+------------------+-----------+-----------+------------------+---------------------------
6a21d09b4076ce51229df8a3 | Famatina         | La Rioja  | Argentina | FAM001           | actualizado@contacto.local
6a21d09b4076ce51229df8a4 | Chilecito        | La Rioja  | Argentina | CHL001           | transito@chilecito.gob.ar 
6a21d09b4076ce51229df8a5 | La Rioja Capital | La Rioja  | Argentina | LRJ001           | transito@larioja.gob.ar   
6a21dfe66d66bffcfa49788d | Olta             | La Rioja  | Argentina | PRB001           | prueba@town.local         
6a21e00d6d66bffcfa49788e | Olta             | La Rioja  | Argentina | PRB001           | prueba@town.local         


In [9]:
# Celda 3 - Insertar municipio con chequeos y mensajes claros
nuevo = {
    "nombre": "San Blas",
    "provincia": "La Rioja",
    "pais": "Argentina",
    "codigo_municipio": "PRB002",
    "contacto": {"email": "prueba@town.local", "telefono": "+54 9 0000 000000"},
    "ubicacion": {"type": "Point", "coordinates": [-67.5, -29.2]}
}

try:
    mid = municipio_dao.insertar(nuevo)
    print("✅ Municipio insertado con _id:", mid)
except ValueError as e:
    print("⚠️ No se insertó el municipio:", e)
except Exception as e:
    print("❌ Error inesperado al insertar municipio:", type(e).__name__, "-", e)


✅ Municipio insertado con _id: 6a21e272bf19a75076c46d07


In [11]:
# Celda 3 - Insertar municipio con chequeos y mensajes claros
nuevo = {
    "nombre": "San Blas",
    "provincia": "La Rioja",
    "pais": "Argentina",
    "codigo_municipio": "PRB002",
    "contacto": {"email": "prueba@town.local", "telefono": "+54 9 0000 000000"},
    "ubicacion": {"type": "Point", "coordinates": [-67.5, -29.2]}
}

try:
    mid = municipio_dao.insertar(nuevo)
    print("✅ Municipio insertado con _id:", mid)
except ValueError as e:
    print("⚠️ No se insertó el municipio:", e)
except Exception as e:
    print("❌ Error inesperado al insertar municipio:", type(e).__name__, "-", e)

⚠️ No se insertó el municipio: Ya existe un municipio con codigo_municipio='PRB002'


In [12]:
# Celda 4 - Obtener por id y actualizar (ejecutar después de insertar/listar)
# Toma el primer municipio disponible si no se provee un _id explícito
municipios = municipio_dao.listar(limit=5)
mid = municipios[0]["_id"] if municipios else None

def print_doc_vertical(doc):
    if not doc:
        print("Documento vacío o no encontrado")
        return
    for k, v in doc.items():
        print(f"{k}: {v}")

if not mid:
    print("No hay municipios disponibles para mostrar/actualizar.")
else:
    print("Municipio seleccionado (_id):", mid)
    print("\nAntes:")
    doc_before = municipio_dao.obtener_por_id(mid)
    print_doc_vertical(doc_before)

    # Ejemplo de cambios: actualizar email de contacto y agregar un campo de nota
    cambios = {
        "contacto": {"email": "actualizado@contacto.local", "telefono": (doc_before.get("contacto") or {}).get("telefono")},
        "nota": "Actualizado desde notebook"
    }

    try:
        mod_count = municipio_dao.actualizar(mid, cambios)
        print("\nCampos modificados (modified_count):", mod_count)
        print("\nDespués:")
        doc_after = municipio_dao.obtener_por_id(mid)
        print_doc_vertical(doc_after)
    except ValueError as e:
        print("No se actualizó el municipio:", e)
    except Exception as e:
        print("Error inesperado al actualizar:", type(e).__name__, "-", e)


Municipio seleccionado (_id): 6a21d09b4076ce51229df8a3

Antes:
_id: 6a21d09b4076ce51229df8a3
nombre: Famatina
provincia: La Rioja
pais: Argentina
codigo_municipio: FAM001
contacto: {'email': 'transito@famatina.gob.ar', 'telefono': '+54 3825 111111'}

Campos modificados (modified_count): 1

Después:
_id: 6a21d09b4076ce51229df8a3
nombre: Famatina
provincia: La Rioja
pais: Argentina
codigo_municipio: FAM001
contacto: {'email': 'actualizado@contacto.local', 'telefono': '+54 3825 111111'}
nota: Actualizado desde notebook


In [13]:
# Celda 5 - Borrar municipio con chequeo de usuarios asociados
# Selecciona el primer municipio disponible si no se provee un _id explícito
municipios = municipio_dao.listar(limit=10)
mid = municipios[0]["_id"] if municipios else None

def print_doc_vertical(doc):
    if not doc:
        print("Documento vacío o no encontrado")
        return
    for k, v in doc.items():
        print(f"{k}: {v}")

if not mid:
    print("No hay municipios disponibles para borrar.")
else:
    print("Municipio seleccionado (_id):", mid)
    doc = municipio_dao.obtener_por_id(mid)
    print("\nDocumento:")
    print_doc_vertical(doc)

    try:
        usuarios_count = municipio_dao.contar_usuarios(mid)
        print("\nUsuarios asociados:", usuarios_count)
        if usuarios_count == 0:
            deleted = municipio_dao.borrar(mid)
            if deleted:
                print("\n✅ Municipio borrado correctamente (deleted_count = 1).")
            else:
                print("\n⚠️ Intento de borrado realizado pero deleted_count = 0 (posible condición de carrera).")
        else:
            print("\n⛔ No se borra: existen usuarios asociados.")
    except ValueError as e:
        print("\n⚠️ Error de validación:", e)
    except Exception as e:
        print("\n❌ Error inesperado al intentar borrar:", type(e).__name__, "-", e)


Municipio seleccionado (_id): 6a21d09b4076ce51229df8a3

Documento:
_id: 6a21d09b4076ce51229df8a3
nombre: Famatina
provincia: La Rioja
pais: Argentina
codigo_municipio: FAM001
contacto: {'email': 'actualizado@contacto.local', 'telefono': '+54 3825 111111'}
nota: Actualizado desde notebook

Usuarios asociados: 4

⛔ No se borra: existen usuarios asociados.


In [14]:
# Celda eliminar municipio por codigo_municipio
codigo = "PRB002"   # ajustá si querés otro código

# Buscar documento
doc = municipio_dao.obtener_por_codigo(codigo)
if not doc:
    print(f"No se encontró ningún municipio con codigo_municipio = {codigo}")
else:
    mid = doc["_id"]
    print("Municipio encontrado:")
    print("  _id:", mid)
    print("  nombre:", doc.get("nombre"))
    print("  provincia:", doc.get("provincia"))
    print("  codigo_municipio:", doc.get("codigo_municipio"))

    try:
        usuarios_count = municipio_dao.contar_usuarios(mid)
        print("Usuarios asociados:", usuarios_count)
        if usuarios_count == 0:
            deleted = municipio_dao.borrar(mid)
            if deleted:
                print("✅ Municipio eliminado correctamente (deleted_count = 1).")
            else:
                print("⚠️ Intento de borrado realizado pero deleted_count = 0.")
        else:
            print("⛔ No se borra: existen usuarios asociados.")
    except ValueError as e:
        print("⚠️ Error de validación:", e)
    except Exception as e:
        print("❌ Error inesperado al intentar borrar:", type(e).__name__, "-", e)


Municipio encontrado:
  _id: 6a21e272bf19a75076c46d07
  nombre: San Blas
  provincia: La Rioja
  codigo_municipio: PRB002
Usuarios asociados: 0
✅ Municipio eliminado correctamente (deleted_count = 1).
